In [ ]:
!pip install dgeb datasets scikit-learn biopython pandas numpy matplotlib

In [ ]:
from datasets import load_dataset

ds =load_dataset("tattabio/convergent_enzymes")
print(ds)

In [ ]:
import pandas as pd

train_df=pd.DataFrame(ds['train'])
test_df=pd.DataFrame(ds['test'])

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")
print(f"Number of EC classes (train): {train_df['Label'].nunique()}")
print(f"Number of EC classes (test): {test_df['Label'].nunique()}")
print(f"Avg sequence length (train): {train_df['Sequence'].str.len().mean():.0f}")
print(f"Avg sequence length (test): {test_df['Sequence'].str.len().mean():.0f}")
print(f"\nSample entry:")
print(train_df.iloc[0])
print(f"\nTop 10 EC classes in train:")
print(train_df['Label'].value_counts().head(10))

In [ ]:
from collections import Counter
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def extract_features(sequence):
    valid_aa=set('ACDEFGHIKLMNPQRSTVWY')
    clean_seq=''.join([aa for aa in sequence.upper() if aa in valid_aa])

    if len(clean_seq)<5:
        return None

    aa_list=list('ACDEFGHIKLMNPQRSTVWY')

    #1. Amino acid composition (20 features)
    aa_counts=Counter(clean_seq)
    total=len(clean_seq)
    aac=[aa_counts.get(aa, 0)/total for aa in aa_list]

    #2. Dipeptide frequency (400 features)
    dipeptides=[a + b for a in aa_list for b in aa_list]
    dp_counts=Counter([clean_seq[i:i+2] for i in range(len(clean_seq)-1)])
    dp_total=len(clean_seq)-1
    dpf=[dp_counts.get(dp, 0)/dp_total for dp in dipeptides]

    #3. Physicochemical descriptors (4 features)
    analysis=ProteinAnalysis(clean_seq)
    physchem=[
        analysis.molecular_weight()/100000,
        analysis.isoelectric_point()/14,
        analysis.gravy(),
        analysis.aromaticity(),
    ]

    return aac+dpf+physchem

#Test on one sequence
test_feat = extract_features(ds['train'][0]['Sequence'])
print(f"Feature vector length: {len(test_feat)}")
print(f"First 10 values: {test_feat[:10]}")

In [ ]:
import numpy as np

X_train=[]
y_train=[]
X_test=[]
y_test=[]

#Extracting features for training set
print("Extracting train features...")
for i, example in enumerate(ds['train']):
    feat=extract_features(example['Sequence'])
    if feat is not None:
        X_train.append(feat)
        y_train.append(example['Label'])

#Extracting features for test set
print("Extracting test features...")
for i, example in enumerate(ds['test']):
    feat=extract_features(example['Sequence'])
    if feat is not None:
        X_test.append(feat)
        y_test.append(example['Label'])

X_train=np.array(X_train)
y_train=np.array(y_train)
X_test=np.array(X_test)
y_test=np.array(y_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

#Scaling features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

#Model 1: SVM
print("Training SVM...")
svm=SVC(kernel='rbf')
svm.fit(X_train_scaled, y_train)
svm_preds=svm.predict(X_test_scaled)
svm_f1=f1_score(y_test, svm_preds, average='weighted')
svm_acc=accuracy_score(y_test, svm_preds)
print(f"SVM - F1: {svm_f1:.3f}, Accuracy: {svm_acc:.3f}")

#Model 2: Random Forest
print("Training Random Forest...")
rf=RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds=rf.predict(X_test)
rf_f1=f1_score(y_test, rf_preds, average='weighted')
rf_acc=accuracy_score(y_test, rf_preds)
print(f"Random Forest - F1: {rf_f1:.3f}, Accuracy: {rf_acc:.3f}")

#Model 3: Logistic Regression
print("Training Logistic Regression...")
lr=LogisticRegression(max_iter=2000)
lr.fit(X_train_scaled, y_train)
lr_preds=lr.predict(X_test_scaled)
lr_f1=f1_score(y_test, lr_preds, average='weighted')
lr_acc=accuracy_score(y_test, lr_preds)
print(f"Logistic Regression - F1: {lr_f1:.3f}, Accuracy: {lr_acc:.3f}")

print(f"\n----- Paper baseline to beat -----")
print(f"ESM2-3B (best FM): F1=0.265")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

#kNN with different k values
for k in [1,3,5]:
    knn=KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    knn_preds=knn.predict(X_test_scaled)
    knn_f1=f1_score(y_test, knn_preds, average='weighted')
    knn_acc=accuracy_score(y_test, knn_preds)
    print(f"kNN (k={k}) - F1: {knn_f1:.3f}, Accuracy: {knn_acc:.3f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Turning sequences into k-mer strings
def seq_to_kmers(sequence, k=3):
    return ' '.join([sequence[i:i+k] for i in range(len(sequence)-k+1)])

#Trying different k values with TF-IDF + nearest centroid approach
for k in [2, 3, 4]:
    train_kmers=[seq_to_kmers(ex['Sequence'], k) for ex in ds['train']]
    test_kmers=[seq_to_kmers(ex['Sequence'], k) for ex in ds['test']]

    tfidf=TfidfVectorizer()
    X_tr_tfidf=tfidf.fit_transform(train_kmers)
    X_te_tfidf=tfidf.transform(test_kmers)

    #For each test sequence, we need to find the closest train sequence
    sim_matrix = cosine_similarity(X_te_tfidf, X_tr_tfidf)

    preds=[]
    for i in range(len(y_test)):
        best_match=sim_matrix[i].argmax()
        preds.append(y_train[best_match])

    f1=f1_score(y_test, preds, average='weighted')
    acc=accuracy_score(y_test, preds)
    print(f"TF-IDF k={k} (nearest neighbor) - F1: {f1:.3f}, Accuracy: {acc:.3f}")

In [ ]:
from sklearn.svm import SVC

results={}

#AAC only (first 20 features)
svm=SVC(kernel='rbf')
svm.fit(scaler.fit_transform(X_train[:, :20]), y_train)
preds=svm.predict(scaler.transform(X_test[:, :20]))
results['AAC only (20 feat)'] = f1_score(y_test, preds, average='weighted')

#Dipeptide only (features 20 to 420)
svm =SVC(kernel='rbf')
svm.fit(scaler.fit_transform(X_train[:, 20:420]), y_train)
preds = svm.predict(scaler.transform(X_test[:, 20:420]))
results['Dipeptide only (400 feat)'] = f1_score(y_test, preds, average='weighted')

#Physicochemical only (last 4 features)
svm =SVC(kernel='rbf')
svm.fit(scaler.fit_transform(X_train[:, 420:]), y_train)
preds =svm.predict(scaler.transform(X_test[:, 420:]))
results['Physicochemical only (4 feat)'] = f1_score(y_test, preds, average='weighted')

#All of them combined
svm= SVC(kernel='rbf')
svm.fit(scaler.fit_transform(X_train), y_train)
preds = svm.predict(scaler.transform(X_test))
results['All combined (424 feat)'] = f1_score(y_test, preds, average='weighted')

print("Feature Ablation Results:")
print("-" * 40)
for name, score in results.items():
    print(f"{name}: F1 = {score:.4f}")

In [ ]:
import matplotlib.pyplot as plt

#All results collected
models={
    'SVM (ours)': 0.009,
    'Random Forest (ours)': 0.014,
    'Logistic Reg (ours)': 0.016,
    'kNN k=1 (ours)': 0.002,
    'TF-IDF NN (ours)': 0.001,
    'ESM2-8M': 0.116,
    'ESM2-35M': 0.201,
    'ESM2-150M': 0.246,
    'ESM2-650M': 0.257,
    'ESM2-3B': 0.265,
    'ProtTrans': 0.243,
    'ProGen2-small': 0.095,
}


print("=" * 50)
print(f"{'Model':<25} {'F1 (weighted)':>15}")
print("=" * 50)
for model, score in models.items():
    marker = " <-- ours" if "(ours)" in model else ""
    print(f"{model:<25} {score:>15.3f}{marker}")
print("=" * 50)


fig, ax = plt.subplots(figsize=(10, 5))

names = list(models.keys())
scores = list(models.values())
colors = ['#534AB7' if '(ours)' in n else '#888780' for n in names]

bars = ax.barh(names, scores, color=colors)
ax.set_xlabel('F1 Score (weighted)')
ax.set_title('Convergent Enzymes Classification: Our Methods vs Foundation Models')
ax.invert_yaxis()

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('convergent_enzymes_comparison.png', dpi=150)
plt.show()
print("\nChart saved as convergent_enzymes_comparison.png")